# 17 — Ensemble v2 no OD (EF01): sazonal + LSTNet-mean (13) + LGBM-nativo + DLinear-res-mean + NNLS por zona
Espelho OD do 16 (ensemble v2 PH). NNLS com pesos nas fatias 1–4, reporte honesto na fatia 5 (dez) + in-sample declarado nas 1–4. LSTNet = média das 5 seeds do 13 (sem retreino). LGBM nativo direto (`save_model`, sem pickle). 2025 intocado.

## Diff exato vs 07 (`notebooks/07-ensemble-od.ipynb`)

| | 07 (v1) | 17 (v2, este notebook) |
|---|---|---|
| Janela | `L=8640 → H=288` (30 d) | **`L=2304 → H=288` (8 d, protocolo v2, idêntico ao 11/13)** |
| Val | 4 fatias por data de fim, sem purge (S1 parcial: 657 janelas) | **5 fatias do 11 + purge/embargo ±H (idêntico ao 11/13): treino 77.141 + val 12.960 ([2880, 2880, 2880, 1440, 2880]) + purge 4.320 + NaN-desc 8.109** |
| LSTNet | 1 seed do 03 (`modelos/lstnet_od.pt`, `in_ch=3`) | **média das 5 seeds do 13 (`modelos/lstnet_od_s{seed}.pt`, `in_ch=8` com ToD+solar+Fourier); erro claro se ausentes** |
| LGBM | 288 `LGBMRegressor` (sklearn) no resíduo, `pickle` (`lgbm_steps.pkl`), 25 base + hora sin/cos (27 cols) | **288 `lgb.train` nativos no resíduo (1 run), `save_model` em `modelos/lgbm_nativo/` (sem pickle); base do 07 a `L=2304` (mesmos lags, ainda válidos: máx 2016 < 2304) + 4 Fourier do dia-da-origem + solar do passo-alvo + hora sin/cos (32 cols; `nomes` atualizado)** |
| DLinear-res | 1 seed (`dlinear_res_od.pt`), early-stopping na val cheia | **×5 seeds (`dlinear_res_od_s{seed}.pt`), mesmo `DLinearLite` do 07, early-stopping só nas fatias 1–4; seed-mean como componente** |
| NNLS | pesos na val cheia (4 fatias), 1 número | **pesos nas fatias 1–4 (`va14[::4]`), reporte honesto na fatia 5 (dez) + in-sample declarado nas 1–4; `ensemble.json` + `normalizacao.json`** |
| Réguas | ens val 0,1325 (v1) | **v1 07 ens 0,1325 (caveat protocolo); v2-11 sazonal 0,1736; v2-13 lstnet (a executar — leitura com skip elegante)** |

## Diff vs 16 (`notebooks/16-v2-ensemble-ph.ipynb` — espelho PH)
16 ainda não existe neste checkout — este notebook segue a especificação travada espelhando o 07 e o protocolo v2 do 11/13, com a variável trocada (pH→OD, `Oxigênio Dissolvido (mg/L)`, micro-outages em vez do outage ~17 d do pH). Quando o 16 existir, o diff esperado é só: CSV/coluna, `OUT=17-v2-ensemble-od`, nomes `*_od_*`, régua v2-10→v2-11 e 12→13. Nada de lógica divergente.

## Protocolo v2 (travado, idêntico ao 11/13)
- `L=2304 → H=288` (8 d → 1 d, passo 5 min), interpolação `time` limite 24, descarte de janelas com NaN.
- Val = 5 fatias por data de fim: 19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]** → val 12.960; treino pós-purge 77.141 + purge 4.320 + NaN-desc 8.109 (OD só tem micro-outages → 5 fatias cheias).
- Purge/embargo: treino exclui janelas cujo alvo `[fim−H, fim]` intersecte qualquer fatia estendida `±H` (gap mín +289 passos). Splitter `purge_train`/`signed_gap_steps` **verbatim** do 11/`validate_split.py` — ver §5 (trava por `assert`).
- Zonas NNLS: `va14` = fatias 1–4 (10.080 origens; fit + in-sample) vs `va5` = fatia 5/dez (2.880 origens; honesto p/ os pesos). DLinear early-stopping só em `va14`. LGBM sem early-stopping (150 rounds fixos, como no 07). Caveat declarado: o LSTNet do 13 usou a val cheia (incl. dez) no early-stopping — a fatia 5 é honesta p/ os pesos NNLS, não p/ a seleção do LSTNet.

## Execução remota (UM job por vez — 12c/23 GB estouram com concorrência; o remoto está com o 13)
- **Solo (máquina livre):** `.venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/17-v2-ensemble-od.ipynb`
- **Compartilhada:** `OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/17-v2-ensemble-od.ipynb`
- Nunca `sleep` dentro do comando remoto (o canal MCP expira); `pkill -f` com o truque `[.]` (ex.: `pkill -f 'nbconvert.*17-v2-ensemble-o[d]'`).
- Estimativa: **~10–20 min solo / ~20–35 min compartilhada** (LGBM 288×~12,9k linhas ≈ 35–45 s; DLinear 5 seeds ≈ 45–90 s; inferência LSTNet 5 seeds + resto vetorizado ≈ 2–5 min; sem treino LSTNet aqui). Rode **após** o 13 (este notebook exige os 5 `lstnet_od_s*.pt`).
- Prophet/ARIMA **não** correm aqui (só os 4 componentes + NNLS).

## Saídas (criadas pela execução)
`resultados/17-v2-ensemble-od/`: `metricas_val.csv` (pooled 5 fatias, referência mista) · `metricas_nnls_zonas.csv` (primária honesta: fit14 in-sample vs holdout5 dez) · `metricas_por_fatia.csv` (5 fatias × modelos, 1 loop, sem groupby duplo) · `metricas_val_diaria.csv` (45 dias-âncora) · `metricas_por_dia.csv` (45 linhas, incl. dez) · `modelos/lgbm_nativo/model_j*.txt` (×288, nativo) + `modelos/dlinear_res_od_s{seed}.pt` (×5) + `modelos/ensemble.json` + `modelos/normalizacao.json` · `figs/` 01-eda/02-limpeza/03-stl/04-forecasts/05-mae/06-val-dias/07-importancia-lgbm (espelho do 07; 04 com LSTNet-mean e banda das 5 seeds). O `README.md` do experimento (formato do 07 + seção “Protocolo v2” + esquema NNLS por zona) é escrito **após** a execução, com números reais + procedência remota (host + work dir).

Convenção: nada in-place em 00–15 · nada de `src/` · nada de 2025 neste notebook (só citação de régua futura; nenhum `dados/benchmark` no código).


In [1]:
import json
import os
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_oxigenio-dissolvido_2024.csv"
OUT = ROOT / "resultados" / "17-v2-ensemble-od"
(OUT / "modelos" / "lgbm_nativo").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# Protocolo v2 (travado, idêntico ao 11/13)
L, H = 2304, 288  # 8 d -> 1 d (passo 5 min)
SEASON = 288
INTERP_LIMIT = 24  # 2 h
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24"),
              ("2024-12-13", "2024-12-22")]
FIT_SLICES = VAL_SLICES[:4]   # NNLS fit + DLinear early-stopping
HOLD_SLICE = VAL_SLICES[4:]   # NNLS holdout honesto (dez)
LN = 2016  # cauda da janela (7 d dos 8 d; mesmo do 07/13)
LGB_EST, LGB_LR, LGB_LEAVES = 150, 0.05, 31  # idêntico ao 07 (1 run, sem early-stopping)
LGB_STRIDE = 6   # idêntico ao 07 (treino LGBM/DLinear)
DL_EPOCHS, DL_PAT = 30, 5     # idêntico ao 07, por seed
ENS_STRIDE = 4   # subsample do fit NNLS nas fatias 1-4 (idêntico ao 07)
TR_INF_STRIDE = 16  # subsample da inferência de treino (idêntico ao 07)
SEEDS = [42, 7, 123, 2024, 999]  # idêntico ao 12/13 (LSTNet-mean + DLinear ×5)
DEVICE = torch.device("cpu")

print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)
print(f"L={L} H={H} LN={LN} fatias={len(VAL_SLICES)} (fit {len(FIT_SLICES)} + holdout {len(HOLD_SLICE)}) seeds={SEEDS}")
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
      "| cpu:", os.cpu_count())
# UM job por vez (12c/23GB estouram com concorrência; remoto ocupado com o 13).
# Estimativa: ~10-20 min solo / ~20-35 min compartilhada (sem treino LSTNet aqui; ver cabeçalho).
print("OUT:", OUT)


ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu
L=2304 H=288 LN=2016 fatias=5 (fit 4 + holdout 1) seeds=[42, 7, 123, 2024, 999]
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>'} | cpu: 12
OUT: /home/marcos/temporal-model/resultados/17-v2-ensemble-od


## 1. Carga

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "Oxigênio Dissolvido (mg/L)": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()


(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 594 (0.6%)


,ds,y
count,105121,104527.000000
mean,2024-07-01 12:00:00,4.491787
min,2024-01-01 00:00:00,0.790000
25%,2024-04-01 06:00:00,2.810000
50%,2024-07-01 12:00:00,4.870000
75%,2024-09-30 18:00:00,6.000000
max,2024-12-31 00:00:00,7.710000
std,NaN,1.725755


## 2. EDA

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("od EF01 2024 — série completa (treino)")
ax[0].set_ylabel("od")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")


maior gap: 334 passos = 27.8 h | gaps > 24 passos: 3


fig salva


## 3. Limpeza

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots = {len(g)*5/60:.1f} h)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")


slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 336
  outage 2024-02-02 13:45:00 → 2024-02-02 14:45:00 (13 slots = 1.1 h)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots = 1.1 h)
  outage 2024-03-25 15:30:00 → 2024-03-26 17:15:00 (310 slots = 25.8 h)
fig salva


## 4. ADF + STL (trecho limpo jul–ago)

In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")


ADF stat=-6.61 p-valor=6.3e-09 → estacionária


fig salva


## 5. Janelamento (L=2304) + val em 5 fatias + purge/embargo ±H + zonas NNLS

Idêntico ao 11/13: janelas por data de **fim**, descarte com NaN pós-interp, treino = janelas válidas fora da val que **sobrevivem ao purge** (alvo `[fim−H, fim]` sem interseção com qualquer fatia estendida `±H`). Funções `purge_train`/`signed_gap_steps` **verbatim** do 11/`validate_split.py` (trava por `assert`). Esperado OD: treino 77.141 + val 12.960 ([2880, 2880, 2880, 1440, 2880]) + purge 4.320 + NaN-desc 8.109 (OD só tem micro-outages → 5 fatias cheias).

Zonas NNLS (novo no ensemble v2): `va14` = fatias 1–4 (10.080 origens; fit dos pesos + in-sample declarado) vs `va5` = fatia 5/dez (2.880 origens; holdout honesto p/ os pesos). Dias-âncora (23:55): 45 = 10+10+10+5+10 (`daily14` 35 vs `daily5` 10).


In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy().astype(np.float32)  # float32: corta a cópia das janelas pela metade (idem 07/11)
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
n_desc_nan = int((~ok).sum())
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date

# --- val por data de fim (5 fatias, incl. dez) ---
is_val = np.zeros(len(ends), dtype=bool)
per_slice = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    per_slice.append(int(m.sum()))
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")


def purge_train(ends, is_val, slices):
    """Descarta treino cujo alvo [fim-H, fim] intersecte fatia estendida ±H.
    Verbatim de /tmp/v2split/validate_split.py (H global)."""
    keep = is_val.copy()
    drop = np.zeros(len(ends), dtype=bool)
    for a, b in slices:
        A = pd.Timestamp(a) - pd.Timedelta(minutes=5 * H)          # ini-H
        B = pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) \
            + pd.Timedelta(minutes=5 * H)                           # fim+H
        tgt0 = ends - pd.Timedelta(minutes=5 * H)
        hit = (~is_val) & (ends >= A) & (tgt0 <= B)  # alvo ∩ [A,B] ≠ ∅
        drop |= hit
    keep[~is_val & ~drop] = True  # treino sobrevivente
    return keep, drop  # keep=True → val ou treino válido


def signed_gap_steps(ends_tr, slices):
    """Distância (passos 5min) do alvo [fim-H,fim] à fatia mais próxima; <0 = overlap.
    Verbatim de /tmp/v2split/validate_split.py."""
    if not len(ends_tr):
        return None
    e = ends_tr.values.astype("datetime64[m]").astype(np.int64)  # min
    best = np.full(len(e), 10 ** 12)
    for a, b in slices:
        A = (pd.Timestamp(a).to_datetime64().astype("datetime64[m]").astype(int))
        B = ((pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
             .to_datetime64().astype("datetime64[m]").astype(int))
        s0 = e - H * 5
        ov = (e >= A) & (s0 <= B)
        gap = np.where(e < A, (A - e) // 5, np.where(s0 > B, (s0 - B) // 5, -(np.minimum(e, B) - np.maximum(s0, A)) // 5 - 1))
        best = np.minimum(best, gap)
    return best


keep, drop = purge_train(ends, is_val, VAL_SLICES)
va = np.where(is_val)[0]
tr = np.where(keep & ~is_val)[0]
print(f"treino (pós-purge): {len(tr)} | val: {len(va)} | "
      f"descartadas (NaN): {n_desc_nan} | purge: {int(drop.sum())}")

# --- asserts de cobertura por fatia (OD: só micro-outages → 5 fatias cheias, idem 11) ---
esperado = [288 * ((pd.Timestamp(b) - pd.Timestamp(a)).days + 1) for a, b in VAL_SLICES]
assert per_slice == esperado, f"cobertura por fatia fora do esperado: {per_slice} vs {esperado}"
assert len(va) == sum(esperado) == 12960, f"val total inesperada: {len(va)}"
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
assert int(drop.sum()) > 0, "purge removeu zero janelas — lógica inativa?"

# --- trava do purge: gap mín ≥ H+1 e overlap zero ---
gaps = signed_gap_steps(ends[tr], VAL_SLICES)
print(f"gap mín alvo-treino→val: +{int(gaps.min())} passos (exigido ≥ {H + 1}); "
      f"treino c/ alvo∩val: {int((gaps < 0).sum())}")
assert int((gaps < 0).sum()) == 0, "LEAKAGE: há alvo de treino dentro da val!"
assert int(gaps.min()) >= H + 1, f"purge/embargo falhou: gap {int(gaps.min())} < {H + 1}"

# --- zonas NNLS: fit (1-4) vs holdout (5/dez) ---
va_ends = ends[va]
is14 = np.zeros(len(va), dtype=bool)
for a, b in FIT_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    is14 |= (va_ends.date >= d0) & (va_ends.date <= d1)
va14 = va[is14]    # 10.080: fit NNLS + in-sample
va5 = va[~is14]    # 2.880: holdout honesto p/ os pesos
print(f"va14 (fit 1-4): {len(va14)} | va5 (holdout dez): {len(va5)}")
assert len(va14) == 10080 and len(va5) == 2880, (len(va14), len(va5))

# --- dias-âncora (23:55) na val: 45 = 10+10+10+5+10 ---
daily_mask = (ends.time == pd.Timestamp("23:55").time()) & is_val
daily_idx = np.where(daily_mask)[0]
por_dia_ct = [int(((ends[daily_idx].date >= pd.Timestamp(a).date()) &
                   (ends[daily_idx].date <= pd.Timestamp(b).date())).sum())
              for a, b in VAL_SLICES]
print("dias-âncora na val:", len(daily_idx), "por fatia:", por_dia_ct)
assert len(daily_idx) == 45, f"dias-âncora inesperados: {len(daily_idx)}"
assert por_dia_ct == [10, 10, 10, 5, 10], f"âncoras por fatia: {por_dia_ct}"
daily14 = np.array([i for i in daily_idx if i in set(va14)], dtype=int)
daily5 = np.array([i for i in daily_idx if i in set(va5)], dtype=int)
print(f"âncoras fit14: {len(daily14)} | âncoras holdout5: {len(daily5)}")
assert len(daily14) == 35 and len(daily5) == 10


fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
fatia 2024-12-13 → 2024-12-22: 2880 janelas válidas
treino (pós-purge): 77141 | val: 12960 | descartadas (NaN): 8109 | purge: 4320
gap mín alvo-treino→val: +289 passos (exigido ≥ 289); treino c/ alvo∩val: 0
va14 (fit 1-4): 10080 | va5 (holdout dez): 2880
dias-âncora na val: 45 por fatia: [10, 10, 10, 5, 10]
âncoras fit14: 35 | âncoras holdout5: 10


## 6. Métricas + baselines de referência + réguas


In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
print("treino rolante:")
print(pd.DataFrame({m: metricas(Ytr, p) for m, p in cheap_preds(Xtr).items()}).T.round(4).to_string())
print("val rolante (5 fatias, protocolo v2):")
print(pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T.round(4).to_string())
print("RÉGUAS: v1 07 ens val 0,1325 (L=8640, 4 fatias, sem purge — caveat protocolo, não comparável direto); "
      "v2-11 sazonal val 0,1736 (mesmo protocolo v2, primária dos baratos).")
# Régua v2-13 (LSTNet 5 seeds): leitura com skip elegante — o 13 pode ainda estar executando.
from pathlib import Path as _P
_r13 = ROOT / "resultados" / "13-v2-lstnet-od" / "metricas_val_media_dp.csv"
if _P(_r13).exists():
    print(pd.read_csv(_r13).to_string())
    print("régua 13 carregada:", _r13)
else:
    print(f"régua 13 ausente ({_r13} não existe — 13 a executar; comparação pendente, sem crash).")


treino rolante:


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.3812  0.5694  8.6819  8.6028
sazonal_naive_288  0.2356  0.3650  6.1279  6.0964
media_movel_288    0.3539  0.4763  8.3817  8.3034
val rolante (5 fatias, protocolo v2):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4280  0.6222  7.7187  7.6334
sazonal_naive_288  0.1736  0.2548  3.3647  3.3567
media_movel_288    0.3631  0.4801  6.6894  6.6394
RÉGUAS: v1 07 ens val 0,1325 (L=8640, 4 fatias, sem purge — caveat protocolo, não comparável direto); v2-11 sazonal val 0,1736 (mesmo protocolo v2, primária dos baratos).
  Unnamed: 0   media      dp
0        MAE  0.1467  0.0017
1       RMSE  0.1996  0.0046
2       MAPE  2.8398  0.0603
3      sMAPE  2.8404  0.0577
régua 13 carregada: /home/marcos/temporal-model/resultados/13-v2-lstnet-od/metricas_val_media_dp.csv


## 7. Covariáveis determinísticas + janelas nativas LN (sem leakage)

Verbatim do 13: `tod_sin/cos` (como no 03/07) + elevação solar/90 + 4 Fourier do dia-do-ano — funções `elevacao_solar`/`fourier_doy` idênticas às do 13/`validate_split.py` (Mogi das Cruzes −23,52/−46,19, UTC−3; futuro conhecido, sem leakage). O LSTNet consome a cauda `LN=2016` da janela `L=2304` (7 d dos 8 d); o LGBM usa as mesmas covariáveis de forma tabular (§8: 4 Fourier do dia-da-origem + solar do passo-alvo). RevIN normaliza só o canal de valor (igual ao 03/07/13); covariáveis entram cruas.


In [8]:
LAT, LON, TZ = -23.52, -46.19, -3  # Mogi das Cruzes; ts locais (UTC-3, sem DST em 2024)


def elevacao_solar(ts, lat=LAT, lon=LON, tz=TZ):
    ts = pd.DatetimeIndex(ts)
    doy = ts.dayofyear.to_numpy() + (ts.hour.to_numpy() + ts.minute.to_numpy() / 60) / 24
    g = 2 * np.pi / 365 * (doy - 1 + (ts.hour.to_numpy() - 12) / 24)
    eq = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                   - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = (ts.hour.to_numpy() * 60 + ts.minute.to_numpy()) + eq + 4 * lon - 60 * tz
    ha = np.radians(tst / 4 - 180)
    cosz = np.sin(np.radians(lat)) * np.sin(decl) + np.cos(np.radians(lat)) * np.cos(decl) * np.cos(ha)
    return 90 - np.degrees(np.arccos(np.clip(cosz, -1, 1)))


def fourier_doy(ts, n=366):
    d = pd.DatetimeIndex(ts).dayofyear.to_numpy()
    return (np.sin(2 * np.pi * d / n), np.cos(2 * np.pi * d / n),
            np.sin(4 * np.pi * d / n), np.cos(4 * np.pi * d / n))


# --- canais ToD idênticos ao 03/07 + solar + fourier (todos float32) ---
TOD_SIN = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
TOD_COS = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
SOLAR = (elevacao_solar(s.index) / 90.0).astype(np.float32)  # /90 determinístico → [-1, 1]
F1, F2, F3, F4 = [a.astype(np.float32) for a in fourier_doy(s.index)]
COV = np.stack([TOD_SIN, TOD_COS, SOLAR, F1, F2, F3, F4], axis=1).astype(np.float32)
N_COV, N_CH = 7, 8  # tod_sin/cos + solar + f1..f4 = 7; + valor = 8 (idêntico ao 13)
assert COV.shape == (len(s), N_COV), COV.shape
print(f"sanity solar meio-dia jan: {float(SOLAR[s.index.get_loc('2024-01-15 12:00')]):+.3f} "
      f"meia-noite: {float(SOLAR[s.index.get_loc('2024-01-15 00:00')]):+.3f}")
print(f"sanity fourier 01/jan: {[round(float(v[0]), 3) for v in (F1, F2, F3, F4)]}")

# --- janelas nativas LN=2016 (mesma construção do 03/13; Tln 7 canais) ---
val5 = s.to_numpy().astype(np.float32)
Wln = sliding_window_view(val5, LN)
Tln = sliding_window_view(COV, LN, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1
print(f"Wln {Wln.shape} Tln {Tln.shape} (esperado (_, {LN}) e (_, {LN}, {N_COV}))")
print(f"janelas nativas válidas: {int((rowln >= 0).sum())}/{len(ends)}")
assert Wln.shape[1] == LN and Tln.shape[1:] == (LN, N_COV), (Wln.shape, Tln.shape)
assert int((rowln >= 0).sum()) == len(ends), "cauda LN fora da grade!"
assert Tln.shape[0] == Wln.shape[0] == len(s) - LN + 1


def monta(idxs):
    ii = np.asarray(idxs); r = rowln[ii]
    return Wln[r], Tln[r], Y[ii].astype(np.float32)


sanity solar meio-dia jan: +0.957 meia-noite: -0.500
sanity fourier 01/jan: [0.017, 1.0, 0.034, 0.999]


Wln (103106, 2016) Tln (103106, 2016, 7) (esperado (_, 2016) e (_, 2016, 7))
janelas nativas válidas: 94421/94421


## 8. LightGBM nativo no resíduo (288 modelos, 1 run, features v2)

`base_feats` do 07 a `L=2304` (código idêntico — os lags máx. 2016 e a fase `L−288·k` continuam válidos) + 4 Fourier do dia-da-origem (`origin = fim − H`, mesma `fourier_doy` do §7) + solar do passo-alvo (`elevacao_solar(target_j)/90`, `target_j = fim − (H−1−j)·5min`) + hora sin/cos do passo (como no 07). Total 32 cols (`nomes` atualizado). Treino nativo `lgb.train` (150 rounds fixos, `num_leaves=31`, `lr=0.05`, sem early-stopping — idêntico ao 07), `save_model` em `modelos/lgbm_nativo/model_j*.txt` (sem pickle).


In [9]:
import lightgbm as lgb

def base_feats(Xb, E):
    # Verbatim do 07 (válido a L=2304: máx lag 2016 < 2304; fase L-288k ≥ 288).
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em

def hour_sincos(em, j):
    # Verbatim do 07: hora (inteira) do passo-alvo j a partir do fim do alvo.
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

def origin_fourier(E):
    # 4 Fourier do dia-da-origem (origin = fim do contexto = fim − H). Estático por janela.
    origin = E - pd.Timedelta(minutes=5*H)
    f1s, f1c, f2s, f2c = fourier_doy(origin)
    return np.stack([f1s, f1c, f2s, f2c], axis=1).astype(np.float32)

def target_solar(E, j):
    # Solar do passo-alvo j (target_j = fim − (H−1−j)·5min). Determinístico, sem leakage.
    tgt = E - pd.to_timedelta((H - 1 - j)*5, unit="min")
    return (elevacao_solar(tgt) / 90.0).astype(np.float32)

tr2 = tr[::LGB_STRIDE]
Xb_tr = X[tr2]
E_tr = ends[tr2]
Ftr, emtr = base_feats(Xb_tr, E_tr)
Ff_tr = origin_fourier(E_tr)  # (N,4), igual p/ os 288 modelos
Str = np.stack([Xb_tr[:, L - SEASON + h] for h in range(H)], axis=1)
Rtr = (Y[tr2] - Str).astype(np.float32)
print(f"features base: {Ftr.shape} + fourier-origem {Ff_tr.shape} + hora/solar do passo | resíduo std: {Rtr.std():.4f}")
assert Ftr.shape[1] == 25 and Ff_tr.shape[1] == 4, (Ftr.shape, Ff_tr.shape)

nomes = ["lag1", "lag2", "lag3", "lag6", "lag12", "lag24", "lag36", "lag72", "lag144",
         "lag287", "lag288", "lag289", "lag576", "lag2016", "seasmean7", "seasstd7",
         "rm12", "rs12", "rm36", "rs36", "rm144", "rs144", "rm288", "rs288", "rm2016",
         "f1_sin_orig", "f1_cos_orig", "f2_sin_orig", "f2_cos_orig",
         "hora_sin", "hora_cos", "solar_alvo"]
assert len(nomes) == 32, len(nomes)
LGB_DIR = OUT / "modelos" / "lgbm_nativo"
LGB_DIR.mkdir(parents=True, exist_ok=True)
params = {"objective": "regression", "num_leaves": LGB_LEAVES, "learning_rate": LGB_LR,
          "verbose": -1, "force_col_wise": True}

boosters, imp_acc = [], []
t0 = time.time()
for j in range(H):
    sh, ch = hour_sincos(emtr, j)
    sol = target_solar(E_tr, j)
    Fj = np.column_stack([Ftr, Ff_tr, sh, ch, sol])
    if j == 0:
        assert Fj.shape[1] == 32, Fj.shape
        print(f"matriz treino j=0: {Fj.shape} (esperado (N, 32))")
    dtr = lgb.Dataset(Fj, label=Rtr[:, j], feature_name=nomes)
    bst = lgb.train(params, dtr, num_boost_round=LGB_EST)
    bst.save_model(str(LGB_DIR / f"model_j{j:03d}.txt"))
    boosters.append(bst)
    imp_acc.append(bst.feature_importance(importance_type="gain"))
    if (j + 1) % 72 == 0:
        print(f"  lgbm {j+1}/{H} ...", flush=True)
print(f"lgbm nativo: {len(boosters)} modelos em {time.time()-t0:.0f}s → {LGB_DIR} (sem pickle)")
assert len(boosters) == H == 288
assert len(list(LGB_DIR.glob('model_j*.txt'))) == 288, "save_model incompleto!"

def prevê_lgbm(idxs):
    ii = np.asarray(idxs)
    Xb = X[ii]; E = ends[ii]
    F, em = base_feats(Xb, E)
    Ff = origin_fourier(E)
    S = np.stack([Xb[:, L - SEASON + h] for h in range(H)], axis=1)
    P = np.empty((len(ii), H), dtype=np.float32)
    for j, bst in enumerate(boosters):
        sh, ch = hour_sincos(em, j)
        sol = target_solar(E, j)
        P[:, j] = S[:, j] + bst.predict(np.column_stack([F, Ff, sh, ch, sol]))
    return P

imp = np.mean(imp_acc, axis=0)
ordem = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([nomes[k] for k in ordem], imp[ordem])
ax.set_title("LightGBM nativo — importância média das features (gain, 288 modelos, v2 32 feats)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-importancia-lgbm.png")
print("top features:", [(nomes[k], round(float(imp[k]), 1)) for k in ordem[:7]])
print("fig salva: 07-importancia-lgbm.png")
del Ftr, Rtr, Str, Xb_tr, Ff_tr


features base: (12857, 25) + fourier-origem (12857, 4) + hora/solar do passo | resíduo std: 0.3650
matriz treino j=0: (12857, 32) (esperado (N, 32))


  lgbm 72/288 ...


  lgbm 144/288 ...


  lgbm 216/288 ...


  lgbm 288/288 ...


lgbm nativo: 288 modelos em 34s → /home/marcos/temporal-model/resultados/17-v2-ensemble-od/modelos/lgbm_nativo (sem pickle)
top features: [('f1_cos_orig', 1804.9), ('rs288', 1572.8), ('rm2016', 1426.9), ('rm288', 1328.2), ('f1_sin_orig', 1160.5), ('lag287', 1027.2), ('lag1', 977.2)]
fig salva: 07-importancia-lgbm.png


## 9. DLinear-res × 5 seeds (seed-mean como componente)

Mesmo `DLinearLite` do 07 (`AvgPool k=25` + `lin_t/lin_s LN→H` + RevIN `γ/β`), cauda `LN=2016` da janela `L=2304`. Treino em `tr[::6]` (idêntico ao 07), early-stopping **só nas fatias 1–4** (`va14[::4]` — a fatia 5/dez fica intocada p/ o NNLS honesto), `DL_EPOCHS=30/PAT=5` por seed, Adam/MSE `lr=1e-3`, `batch=512`. Salva `modelos/dlinear_res_od_s{seed}.pt` (×5); o componente do ensemble é a **média das 5 seeds**.


In [10]:
class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, H)
        self.lin_s = nn.Linear(LN, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg

def snaive(X_):
    return np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)

def monta_res(idxs):
    ii = np.asarray(idxs)
    return X[ii][:, -LN:].astype(np.float32), (Y[ii] - snaive(X[ii])).astype(np.float32)

# Early-stopping SÓ nas fatias 1-4 (va14) — va5/dez intocada.
Xr_tr, Rr_tr = monta_res(tr[::LGB_STRIDE])
Xr_fit, Rr_fit = monta_res(va14[::ENS_STRIDE])
print(f"residual: treino {Xr_tr.shape} fit-NNLS(1-4) {Xr_fit.shape} | holdout5 intocado ({len(va5)} origens)")
dl_hists, dl_bests = {}, {}
t_all = time.time()
for SEED in SEEDS:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    dlres = DLinearLite().to(DEVICE)
    opt = torch.optim.Adam(dlres.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_tr), torch.from_numpy(Rr_tr)), batch_size=512, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_fit), torch.from_numpy(Rr_fit)), batch_size=512)
    best, patience, hist = float("inf"), 0, {"train": [], "val14": []}
    t0 = time.time()
    for ep in range(1, DL_EPOCHS + 1):
        dlres.train()
        for xb, yb in tr_loader:
            opt.zero_grad(); loss = loss_fn(dlres(xb), yb); loss.backward(); opt.step()
        dlres.eval(); tl, vl = 0.0, 0.0
        with torch.no_grad():
            for xb, yb in tr_loader:
                tl += float(loss_fn(dlres(xb), yb)) * len(xb)
            for xb, yb in va_loader:
                vl += float(loss_fn(dlres(xb), yb)) * len(xb)
        tl /= len(tr_loader.dataset); vl /= len(va_loader.dataset)
        hist["train"].append(tl); hist["val14"].append(vl)
        tag = ""
        if vl < best:
            best, patience = vl, 0
            torch.save({"state": dlres.state_dict(), "seed": SEED, "ln": LN},
                       OUT / "modelos" / f"dlinear_res_od_s{SEED}.pt")
            tag = " *"
        else:
            patience += 1
        print(f"[dlres s{SEED}] ep {ep:02d} train={tl:.5f} val14={vl:.5f}{tag}", flush=True)
        if patience >= DL_PAT:
            break
    dl_hists[SEED], dl_bests[SEED] = hist, best
    print(f"[dlres s{SEED}] em {time.time()-t0:.0f}s | melhor val14={best:.5f}")
print(f"dlinear-res 5 seeds em {time.time()-t_all:.0f}s")
del Xr_tr, Rr_tr, Xr_fit, Rr_fit

dlres_bank = {}
for SEED in SEEDS:
    m = DLinearLite().to(DEVICE)
    ck = torch.load(OUT / "modelos" / f"dlinear_res_od_s{SEED}.pt", map_location="cpu", weights_only=False)
    m.load_state_dict(ck["state"]); m.eval()
    dlres_bank[SEED] = m
print("dlinear-res checkpoints:", sorted(dlres_bank))

@torch.no_grad()
def prevê_dlres_mean(idxs, batch=256):
    # Seed-mean das 5 seeds (componente Dr do ensemble).
    ii = np.asarray(idxs)
    Ps = snaive(X[ii])
    Xt = torch.from_numpy(X[ii][:, -LN:].astype(np.float32))
    acc = np.zeros((len(ii), H), dtype=np.float32)
    for SEED, m in dlres_bank.items():
        m.eval(); outs = []
        for b in range(0, len(Xt), batch):
            outs.append(m(Xt[b:b+batch]).numpy())
        acc += Ps + np.concatenate(outs)
    return acc / len(dlres_bank)


residual: treino (12857, 2016) fit-NNLS(1-4) (2520, 2016) | holdout5 intocado (2880 origens)


[dlres s42] ep 01 train=0.10107 val14=0.04265 *


[dlres s42] ep 02 train=0.09814 val14=0.04156 *


[dlres s42] ep 03 train=0.09337 val14=0.03942 *


[dlres s42] ep 04 train=0.09503 val14=0.04179


...

[dlres s999] ep 10 train=0.08986 val14=0.03886


[dlres s999] ep 11 train=0.08849 val14=0.03777


[dlres s999] ep 12 train=0.08926 val14=0.04100


[dlres s999] ep 13 train=0.08677 val14=0.03675


[dlres s999] em 14s | melhor val14=0.03598
dlinear-res 5 seeds em 77s
dlinear-res checkpoints: [7, 42, 123, 999, 2024]


## 10. LSTNet-mean (13) + NNLS por zona (pesos 1–4, honesto dez)

`LSTNet1D` verbatim do 13 (`Conv1d 8→32/k12/s6 · GRU 64 · skip GRUCell 32/p48 · head (64+32)→288 · AR 288→288 · dropout 0,1`, RevIN por janela). Componente `Pn` = **média das 5 seeds** de `resultados/13-v2-lstnet-od/modelos/lstnet_od_s{seed}.pt` (**erro claro se ausentes — rode o 13 antes**). NNLS (`scipy.optimize.nnls`, sem intercepto, pesos ≥ 0) ajustado em `va14[::4]` (fatias 1–4); reporte **honesto em `va5` (dez)** + **in-sample declarado em `va14`** + pooled de referência. Salva `modelos/ensemble.json` + `modelos/normalizacao.json`. Caveat: o LSTNet do 13 viu dez no early-stopping — dez é honesto p/ os pesos, não p/ a seleção do LSTNet.


In [11]:
class LSTNet1D(nn.Module):
    # Verbatim do 13 (só in_channels=8: valor + 7 covariáveis do §7).
    def __init__(self, n_ch=8):
        super().__init__()
        self.conv = nn.Conv1d(n_ch, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

CKPT13 = ROOT / "resultados" / "13-v2-lstnet-od" / "modelos"
ausentes = [str(CKPT13 / f"lstnet_od_s{sd}.pt") for sd in SEEDS if not (CKPT13 / f"lstnet_od_s{sd}.pt").exists()]
assert not ausentes, "rode o 13-v2-lstnet-od antes! checkpoints ausentes: " + "; ".join(ausentes)
ruler_bank = {}
for SEED in SEEDS:
    m = LSTNet1D(n_ch=N_CH).to(DEVICE)
    ck = torch.load(CKPT13 / f"lstnet_od_s{SEED}.pt", map_location="cpu", weights_only=False)
    m.load_state_dict(ck["state"]); m.eval()
    ruler_bank[SEED] = m
print("régua 13 recarregada (5 seeds):", sorted(ruler_bank))

@torch.no_grad()
def prevê_lstnet_mean(idxs, batch=256):
    ii = np.asarray(idxs)
    acc = None
    for sd, m in ruler_bank.items():
        m.eval(); outs = []
        for b in range(0, len(ii), batch):
            xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
            tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
            outs.append(m(xb, tb).numpy())
        P = np.concatenate(outs)
        acc = P if acc is None else acc + P
    return acc / len(ruler_bank)

@torch.no_grad()
def prevê_tudo(idxs, batch=256):
    # 4 componentes: Ps (sazonal-288) · Pn (lstnet-mean 13) · Gb (lgbm-nativo) · Dr (dlres-mean).
    Ps = snaive(X[np.asarray(idxs)])
    Gb = prevê_lgbm(idxs)
    Dr = prevê_dlres_mean(idxs, batch=batch)
    Pn = prevê_lstnet_mean(idxs, batch=batch)
    return Ps, Pn, Gb, Dr

from scipy.optimize import nnls
va_fit = va14[::ENS_STRIDE]  # fit NNLS só nas fatias 1-4 (subsample ×4, idem 07)
print(f"fit NNLS: {len(va_fit)} origens nas fatias 1-4 (de {len(va14)}); holdout: {len(va5)} em dez")
Ps_f, Pn_f, Gb_f, Dr_f = prevê_tudo(va_fit)
A = np.column_stack([Ps_f.ravel(), Pn_f.ravel(), Gb_f.ravel(), Dr_f.ravel()])
w, _ = nnls(A, Y[va_fit].ravel())
pesos = {k: round(float(v), 4) for k, v in zip(["sazonal", "lstnet", "lgbm", "dlres"], w)}
print("pesos ensemble (nnls nas fatias 1-4):", pesos, f"soma={float(w.sum()):.4f}")
json.dump({"pesos": pesos, "mode": "nnls-ensemble sobre sazonal+lstnet-mean13+lgbm-nativo+dlres-mean",
           "fit_slices": FIT_SLICES, "holdout_slice": HOLD_SLICE[-1],
           "val_slices": VAL_SLICES, "stride_fit": ENS_STRIDE, "seeds": SEEDS,
           "lgbm": {"est": LGB_EST, "lr": LGB_LR, "leaves": LGB_LEAVES, "features": nomes},
           "note": "pesos ajustados nas fatias 1-4; fatia 5 (dez) = holdout honesto p/ os pesos"},
          open(OUT / "modelos" / "ensemble.json", "w"))
json.dump({"mode": "lgbm-nativo-288 + dlinear-residual-5seeds + lstnet-mean13 + nnls-por-zona",
           "L": L, "H": H, "LN": LN, "in_channels": N_CH, "channels": ["valor", "tod_sin", "tod_cos", "solar Elev/90", "f1_sinA", "f1_cosA", "f2_sinS", "f2_cosS"],
           "solar_scale": 90.0, "lat_lon_tz": [LAT, LON, TZ],
           "val_slices": VAL_SLICES, "fit_slices": FIT_SLICES, "holdout_slice": HOLD_SLICE[-1], "seeds": SEEDS},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("ensemble.json + normalizacao.json salvos")

def ensemble(Ps, Pn, Gb, Dr):
    return w[0]*Ps + w[1]*Pn + w[2]*Gb + w[3]*Dr

t0 = time.time()
tr3 = tr[::TR_INF_STRIDE]
Ps_tr, Pn_tr, Gb_tr, Dr_tr = prevê_tudo(tr3); En_tr = ensemble(Ps_tr, Pn_tr, Gb_tr, Dr_tr)
Ps_14, Pn_14, Gb_14, Dr_14 = prevê_tudo(va14); En_14 = ensemble(Ps_14, Pn_14, Gb_14, Dr_14)
Ps_5, Pn_5, Gb_5, Dr_5 = prevê_tudo(va5); En_5 = ensemble(Ps_5, Pn_5, Gb_5, Dr_5)
Ps_va, Pn_va, Gb_va, Dr_va = prevê_tudo(va); En_va = ensemble(Ps_va, Pn_va, Gb_va, Dr_va)
Ps_d, Pn_d, Gb_d, Dr_d = prevê_tudo(daily_idx); En_d = ensemble(Ps_d, Pn_d, Gb_d, Dr_d)
print(f"inferência em {time.time()-t0:.0f}s")
print("ENS treino (subsample ×16, referência):", {k: round(v, 4) for k, v in metricas(Y[tr3], En_tr).items()})
print("ENS fit14 (IN-SAMPLE 1-4, declarado):", {k: round(v, 4) for k, v in metricas(Y[va14], En_14).items()})
print("ENS holdout5 dez (HONESTO p/ pesos):", {k: round(v, 4) for k, v in metricas(Y[va5], En_5).items()})
print("ENS val pooled 5 fatias (misto, referência):", {k: round(v, 4) for k, v in metricas(Yva, En_va).items()})


régua 13 recarregada (5 seeds): [7, 42, 123, 999, 2024]
fit NNLS: 2520 origens nas fatias 1-4 (de 10080); holdout: 2880 em dez


pesos ensemble (nnls nas fatias 1-4): {'sazonal': 0.0038, 'lstnet': 0.6771, 'lgbm': 0.0, 'dlres': 0.3216} soma=1.0025
ensemble.json + normalizacao.json salvos


inferência em 91s
ENS treino (subsample ×16, referência): {'MAE': 0.1608, 'RMSE': 0.2489, 'MAPE': 4.2586, 'sMAPE': 4.2049}
ENS fit14 (IN-SAMPLE 1-4, declarado): {'MAE': 0.1234, 'RMSE': 0.1688, 'MAPE': 2.4242, 'sMAPE': 2.4301}
ENS holdout5 dez (HONESTO p/ pesos): {'MAE': 0.1733, 'RMSE': 0.2383, 'MAPE': 3.2189, 'sMAPE': 3.1697}


ENS val pooled 5 fatias (misto, referência): {'MAE': 0.1345, 'RMSE': 0.1865, 'MAPE': 2.6008, 'sMAPE': 2.5945}


## 11. Tabelas (zonas honestas + por fatia com 1 loop, sem groupby duplo)
`metricas_nnls_zonas.csv` = primária (fit14 in-sample declarado vs holdout5 dez honesto p/ os pesos). `metricas_val.csv` = pooled 5 fatias (referência mista, como no 07). `metricas_por_fatia.csv` = 5 fatias × modelos via **1 loop explícito (sem `groupby`; proibido groupby duplo)**. Dias-âncora: 45 linhas.


In [12]:
# --- zonas NNLS (primária honesta) ---
zonas = {
    "fit14_in_sample": (va14, (Ps_14, Pn_14, Gb_14, Dr_14, En_14)),
    "holdout5_dez_honesto": (va5, (Ps_5, Pn_5, Gb_5, Dr_5, En_5)),
}
rows_z = []
for zona, (idx_z, (Ps_z, Pn_z, Gb_z, Dr_z, En_z)) in zonas.items():
    Yz = Y[idx_z]
    for nome, Pz in [("sazonal_naive_288", Ps_z), ("lstnet_mean13", Pn_z), ("lgbm", Gb_z),
                     ("dlres_mean", Dr_z), ("ens", En_z)]:
        mm = metricas(Yz, Pz)
        rows_z.append({"zona": zona, "modelo": nome, **mm})
tab_z = pd.DataFrame(rows_z, columns=["zona", "modelo", "MAE", "RMSE", "MAPE", "sMAPE"]).round(4)
tab_z.to_csv(OUT / "metricas_nnls_zonas.csv", index=False)
print("=== zonas NNLS (MAE) ===")
print(tab_z.pivot(index="zona", columns="modelo", values="MAE").to_string())

# --- pooled 5 fatias (referência mista, formato do 07) ---
linhas = {m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}
linhas["lstnet_mean13"] = metricas(Yva, Pn_va)
linhas["lgbm"] = metricas(Yva, Gb_va)
linhas["dlres_mean"] = metricas(Yva, Dr_va)
linhas["ens"] = metricas(Yva, En_va)
tab_va = pd.DataFrame(linhas).T.round(4)
tab_va.to_csv(OUT / "metricas_val.csv")
print("=== val pooled 5 fatias (misto) ===")
print(tab_va.to_string())

# --- por fatia: 1 loop explícito, sem groupby (proibido groupby duplo) ---
rows_f = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m_va = (va_ends.date >= d0) & (va_ends.date <= d1)
    pos = np.where(m_va)[0]  # posições dentro de va
    loc = va[m_va]            # índices globais
    Ps_l, Pn_l, Gb_l, Dr_l = Ps_va[pos], Pn_va[pos], Gb_va[pos], Dr_va[pos]
    En_l = ensemble(Ps_l, Pn_l, Gb_l, Dr_l)
    Yl = Y[loc]
    cp_l = cheap_preds(X[loc])
    for nome, Pl in [("persistencia", cp_l["persistencia"]), ("sazonal_naive_288", cp_l["sazonal_naive_288"]),
                     ("media_movel_288", cp_l["media_movel_288"]), ("lstnet_mean13", Pn_l),
                     ("lgbm", Gb_l), ("dlres_mean", Dr_l), ("ens", En_l)]:
        mm = metricas(Yl, Pl)
        rows_f.append({"fatia": f"{a}→{b}", "modelo": nome, **mm})
tab_f = pd.DataFrame(rows_f, columns=["fatia", "modelo", "MAE", "RMSE", "MAPE", "sMAPE"]).round(4)
tab_f.to_csv(OUT / "metricas_por_fatia.csv", index=False)
assert tab_f["fatia"].nunique() == 5, "por_fatia sem as 5 fatias!"
assert (tab_f["fatia"] == "2024-12-13→2024-12-22").any(), "fatia dez ausente no por_fatia!"
print("=== val por fatia (MAE) ===")
print(tab_f.pivot(index="fatia", columns="modelo", values="MAE").to_string())

# --- dias-âncora (45) ---
Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]}
diario["lstnet_mean13"] = metricas(Yd, Pn_d)
diario["lgbm"] = metricas(Yd, Gb_d)
diario["dlres_mean"] = metricas(Yd, Dr_d)
diario["ens"] = metricas(Yd, En_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_val_diaria.csv")
print("=== val dias-âncora (45) ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["lstnet_mean13"] = [mae(Yd[k:k+1], Pn_d[k:k+1]) for k in range(len(Yd))]
por_dia["lgbm"] = [mae(Yd[k:k+1], Gb_d[k:k+1]) for k in range(len(Yd))]
por_dia["dlres_mean"] = [mae(Yd[k:k+1], Dr_d[k:k+1]) for k in range(len(Yd))]
por_dia["ens"] = [mae(Yd[k:k+1], En_d[k:k+1]) for k in range(len(Yd))]
por_dia.insert(0, "fatia", [next(f"{a}→{b}" for a, b in VAL_SLICES
              if pd.Timestamp(a).date() <= ends[daily_idx[k]].date() <= pd.Timestamp(b).date())
              for k in range(len(Yd))])
por_dia.to_csv(OUT / "metricas_por_dia.csv")
assert len(por_dia) == 45 and (por_dia["fatia"] == "2024-12-13→2024-12-22").sum() == 10
print(por_dia.round(4).to_string())
print(f"\nMelhor pooled (referência mista): {tab_va['MAE'].idxmin()} = {tab_va['MAE'].min():.4f}")
print(f"Holdout5 dez honesto (primário p/ pesos): ens = {tab_z[(tab_z.zona=='holdout5_dez_honesto')&(tab_z.modelo=='ens')]['MAE'].iloc[0]:.4f}")
print(f"pesos ensemble (fit 1-4): {pesos}")


=== zonas NNLS (MAE) ===
modelo                dlres_mean     ens    lgbm  lstnet_mean13  sazonal_naive_288
zona                                                                              
fit14_in_sample           0.1336  0.1234  0.1795         0.1262             0.1518
holdout5_dez_honesto      0.1842  0.1733  0.2636         0.1848             0.2499


=== val pooled 5 fatias (misto) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4280  0.6222  7.7187  7.6334
sazonal_naive_288  0.1736  0.2548  3.3647  3.3567
media_movel_288    0.3631  0.4801  6.6894  6.6394
lstnet_mean13      0.1392  0.1902  2.6920  2.6923
lgbm               0.1982  0.2660  3.7272  3.7950
dlres_mean         0.1448  0.2050  2.7708  2.7695
ens                0.1345  0.1865  2.6008  2.5945


=== val por fatia (MAE) ===
modelo                 dlres_mean     ens    lgbm  lstnet_mean13  media_movel_288  persistencia  sazonal_naive_288
fatia                                                                                                             
2024-04-19→2024-04-28      0.0990  0.0992  0.1203         0.1042           0.1615        0.1536             0.1240
2024-07-20→2024-07-29      0.1146  0.1093  0.2210         0.1154           0.1381        0.1429             0.1316
2024-09-15→2024-09-24      0.1792  0.1581  0.1589         0.1535           0.4498        0.5584             0.1969
2024-11-20→2024-11-24      0.1493  0.1304  0.2558         0.1368           0.5478        0.6852             0.1579
2024-12-13→2024-12-22      0.1842  0.1733  0.2636         0.1848           0.6106        0.7283             0.2499
=== val dias-âncora (45) ===
                      MAE    RMSE     MAPE   sMAPE
persistencia       0.5386  0.7313  10.3258  9.5216
sazonal_naive_288  0.1748  0.2572   

## 12. Figuras (espelho do 07 + bandas das 5 seeds e zonas)


In [13]:
ks = [0, len(tr3) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
cp = cheap_preds(X[tr3])
for ax, k in zip(axes, ks):
    tf = pd.date_range(ends[tr3[k]] - pd.Timedelta(minutes=5*(H-1)), ends[tr3[k]], freq="5min")
    ax.plot(tf, Y[tr3[k]], "k-", lw=1.5, label="real")
    ax.plot(tf, cp["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Pn_tr[k], lw=1, alpha=0.6, label="lstnet-mean13")
    ax.plot(tf, Gb_tr[k], lw=1, alpha=0.9, label="lgbm-nativo")
    ax.plot(tf, En_tr[k], lw=1.2, alpha=0.9, label="ens (pesos 1-4)")
    ax.set_title(f"origem {ends[tr3[k]]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab_va["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE na val pooled 5 fatias (misto; menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, ax = plt.subplots(figsize=(12, 3.5))
for col, ls in [("sazonal_naive_288", "--"), ("lstnet_mean13", "-."), ("ens", "-"), ("lgbm", ":"), ("persistencia", ":")]:
    if col in por_dia.columns:
        ax.plot(pd.to_datetime(por_dia.index), por_dia[col], ls, lw=1.1, label=col)
ax.axvspan(pd.Timestamp("2024-12-13"), pd.Timestamp("2024-12-22"), color="grey", alpha=0.15, label="holdout dez (honesto)")
ax.set_title("od — MAE por dia-âncora na val (5 fatias v2; faixa = holdout honesto)")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")
print("figs salvas")


figs salvas


## 13. Conclusões (preencher com números reais após a execução)

Réguas v2 acima (`metricas_nnls_zonas.csv` = primária honesta: `holdout5_dez_honesto` p/ os pesos vs `fit14_in_sample` declarado; `metricas_por_fatia.csv` mostra cada fatia, incl. dez). Régua v1 de referência: **07 ens val 0,1325** (L=8640, 4 fatias sem purge — caveat protocolo, não comparável direto). Régua v2 dos baratos: **11 sazonal val 0,1736** (mesmo protocolo). Régua v2-13 LSTNet: ler `resultados/13-v2-lstnet-od/metricas_val_media_dp.csv` após o 13 (skip elegante no §6 se ainda ausente). Caveat declarado: o LSTNet do 13 viu dez no early-stopping — dez é honesto p/ os pesos NNLS, não p/ a seleção do LSTNet. Checkpoints em `modelos/` (`lgbm_nativo/` + `dlinear_res_od_s*.pt`; `lstnet` reusado do 13) para o benchmark futuro.

### Protocolo v2 (resumo p/ o README do experimento)
- Janelas `L=2304 → H=288` (8 d → 1 d, 5 min), interp `time` limite 24, descarte com NaN; val 5 fatias por data de fim (19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]**); purge/embargo ±H (gap mín +289; trava por `assert`).
- Componentes: sazonal-288 + LSTNet-mean (5 seeds do 13, `in_ch=8`) + LGBM nativo 288 (1 run, 32 feats: base do 07 + 4 Fourier origem + solar alvo) + DLinear-res-mean (5 seeds, early-stopping só 1–4).
- NNLS nas fatias 1–4 (`va14[::4]`), honesto em dez (`va5`), in-sample declarado em 1–4; `ensemble.json` + `normalizacao.json`.
- OD só tem micro-outages (336 slots NaN pós-interp) → 5 fatias cheias, asserts idênticos aos do 11/13.

### Procedência da execução (preencher no commit da execução)
- Host remoto: `temporal-remote` 192.168.1.6 · work dir: `/home/marcos/temporal-model` · data: `2026-09-17` · pré-requisito: 13 executado (5 `lstnet_od_s*.pt`)
- Pós-execução: escrever `resultados/17-v2-ensemble-od/README.md` (formato do 07 + seção “Protocolo v2” + esquema NNLS por zona), indexar em `resultados/README.md` + `notebooks/README.md` + README §7 — com números reais. Não commitar `modelos/lgbm_nativo/*.txt` nem `modelos/*.pt` (vão ao Release via `scripts/baixar_modelos.sh`).
